# Two stages on three buses

Layer 2: [Dispatch](../subsystems/dispatch.md). Layer 3: [scripts/market](../modules/market.md).

The two-stage market first clears a GB wholesale market with no internal network limits (Stage 1), then moves units
off those positions until the dispatch fits the network (Stage 2, the balancing mechanism). This notebook runs the
idea on three GB buses and one foreign bus, using the model's own copperplate: `scripts.market.copperplate`
collapses the synchronous mainland onto one hub bus and writes the results back onto the original network. It needs
only PyPSA and HiGHS.

The toy has cheap wind in the north, demand in the south, and a line between them that is too small for the wind.

In [1]:
import logging, sys, warnings
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Snakefile").exists())
for path in (ROOT, ROOT / "config"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
warnings.filterwarnings("ignore")
logging.getLogger("pypsa").setLevel(logging.ERROR)
logging.getLogger("linopy").setLevel(logging.ERROR)
logging.getLogger("scripts").setLevel(logging.WARNING)

import pandas as pd
pd.set_option("display.width", 120)

In [2]:
import pypsa
from scripts.market.copperplate import collapse_mainland, expand_results

def toy_network():
    snapshots = pd.date_range("2035-01-01", periods=4, freq="h")
    n = pypsa.Network()
    n.set_snapshots(snapshots)
    for name, x in [("north", 0.0), ("middle", 1.0), ("south", 2.0)]:
        n.add("Bus", name, carrier="AC", country="GB", v_nom=400.0, x=x, y=0.0)
    n.add("Bus", "NO External", carrier="AC", country="NO", v_nom=400.0)
    n.add("Line", "north-middle", bus0="north", bus1="middle", x=0.1, r=0.01, s_nom=600.0)
    n.add("Line", "middle-south", bus0="middle", bus1="south", x=0.1, r=0.01, s_nom=600.0)
    n.add("Line", "north-south", bus0="north", bus1="south", x=0.2, r=0.02, s_nom=300.0)
    wind = pd.Series([0.9, 0.8, 0.5, 0.2], index=snapshots)
    n.add("Generator", "north wind", bus="north", carrier="wind", p_nom=1500.0, p_max_pu=wind, marginal_cost=0.0)
    n.add("Generator", "south gas", bus="south", carrier="gas", p_nom=1500.0, marginal_cost=80.0)
    n.add("Generator", "middle gas", bus="middle", carrier="gas", p_nom=500.0, marginal_cost=95.0)
    n.add("Load", "south demand", bus="south", p_set=pd.Series([1000.0, 1100.0, 1200.0, 1100.0], index=snapshots))
    n.add("Generator", "Norway", bus="NO External", carrier="EU_market", p_nom=700.0, marginal_cost=60.0)
    n.add("Link", "NSL", bus0="NO External", bus1="north", p_nom=700.0, p_min_pu=-1.0)
    return n

network = toy_network()
network.buses[["carrier", "country", "x"]]

,carrier,country,x
name,,,
north,AC,GB,0.0
middle,AC,GB,1.0
south,AC,GB,2.0
NO External,AC,NO,0.0


## Stage 1: the copperplate

`collapse_mainland` finds the largest synchronous sub-network among the non-External buses, adds a hub bus
`GB_mainland`, moves every generator and load on the mainland to it, and removes the mainland's lines. The foreign
bus and the interconnector keep their identity and their limit, so the collapse is exact: with unlimited internal
ratings every mainland bus would share one price anyway.

In [3]:
collapsed, mapping = collapse_mainland(network)
print("hub:", mapping.hub)
print("mainland buses:", mapping.mainland_buses)
print("lines removed:", mapping.removed_lines)
collapsed.buses[["carrier", "country"]]

hub: GB_mainland
mainland buses: ['middle', 'north', 'south']
lines removed: ['north-middle', 'middle-south', 'north-south']


,carrier,country
name,,
NO External,AC,NO
GB_mainland,AC,GB


In [4]:
collapsed.optimize(solver_name="highs", solver_options={"output_flag": False})
expand_results(network, collapsed, mapping)
positions = network.generators_t.p.copy()
wholesale_cost = (positions * network.generators.marginal_cost).sum().sum()
print(f"Stage 1 cost: {wholesale_cost:,.0f} GBP")
print("GB price (GBP/MWh), one value per hour on every mainland bus:")
network.buses_t.marginal_price.round(1) + 0.0

Running HiGHS 1.16.0 (git hash: f3f113d): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
Stage 1 cost: 77,000 GBP
GB price (GBP/MWh), one value per hour on every mainland bus:


name,north,middle,south,NO External
snapshot,,,,
2035-01-01 00:00:00,0.0,0.0,0.0,0.0
2035-01-01 01:00:00,0.0,0.0,0.0,0.0
2035-01-01 02:00:00,60.0,60.0,60.0,60.0
2035-01-01 03:00:00,80.0,80.0,80.0,60.0


The schedule uses all the wind. On the real network it cannot all reach the south: see how loaded the lines would
be if the schedule were flowed.

In [5]:
flowed = toy_network()
flowed.generators_t.p_set = positions
flowed.links_t.p_set = network.links_t.p0
flowed.lpf()
(flowed.lines_t.p0.abs() / flowed.lines.s_nom).round(2).rename(columns=lambda c: f"{c} loading")

name,north-middle loading,middle-south loading,north-south loading
snapshot,,,
2035-01-01 00:00:00,0.83,0.83,1.67
2035-01-01 01:00:00,0.92,0.92,1.83
2035-01-01 02:00:00,1.00,1.00,2.00
2035-01-01 03:00:00,0.83,0.83,1.67


Loadings above 1 are what Stage 2 has to remove.

## Stage 2: redispatch on the network

Here Stage 2 is the network-constrained least-cost dispatch, and the redispatch is the difference from the Stage 1
positions. The model's balancing stage (`scripts/market/solve_balancing.py`) does more: it anchors every unit to its
Stage 1 position and prices each move at the unit's offer (turn up) or bid (turn down) rather than its marginal cost,
with bids for supported plant set by what the support scheme pays (`scripts/market/cfd_bm_bids.py`). The cost here is
therefore the resource cost of the constraint, not a bid-offer cost.

In [6]:
constrained = toy_network()
constrained.optimize(solver_name="highs", solver_options={"output_flag": False})
physical = constrained.generators_t.p
redispatch = (physical - positions).round(1)
print("Redispatch, MW (positive: turned up; negative: turned down)")
redispatch

Redispatch, MW (positive: turned up; negative: turned down)


name,north wind,south gas,middle gas,Norway
snapshot,,,,
2035-01-01 00:00:00,-400.0,400.0,0.0,0.0
2035-01-01 01:00:00,-500.0,500.0,0.0,0.0
2035-01-01 02:00:00,-150.0,600.0,0.0,-450.0
2035-01-01 03:00:00,0.0,400.0,0.0,-400.0


In [7]:
constrained_cost = (physical * constrained.generators.marginal_cost).sum().sum()
summary = pd.Series({
    "Stage 1 cost (GBP)": wholesale_cost,
    "network-constrained cost (GBP)": constrained_cost,
    "constraint cost (GBP)": constrained_cost - wholesale_cost,
    "wind turned down (MWh)": -redispatch["north wind"].clip(upper=0).sum(),
}).round(0)
summary.to_frame("toy")

,toy
Stage 1 cost (GBP),77000.0
network-constrained cost (GBP),178000.0
constraint cost (GBP),101000.0
wind turned down (MWh),1050.0


In [8]:
print("Nodal prices after redispatch (GBP/MWh): the north is priced by what it can export")
constrained.buses_t.marginal_price[["north", "middle", "south"]].round(1) + 0.0

Nodal prices after redispatch (GBP/MWh): the north is priced by what it can export


name,north,middle,south
snapshot,,,
2035-01-01 00:00:00,0.0,40.0,80.0
2035-01-01 01:00:00,0.0,40.0,80.0
2035-01-01 02:00:00,0.0,40.0,80.0
2035-01-01 03:00:00,60.0,70.0,80.0


The pattern is the one the GB studies report at scale: wind in the north turned down, gas in the south turned up,
and a constraint cost that the single GB price of Stage 1 does not show. See [Dispatch](../../model/dispatch/index.md) for the
method, and [tutorial 5](../../tutorials/5-two-stage-market-2030.ipynb) for a real week.